# Local-to-particle evaluation (L2P)

## Purpose

L2P evaluates a target-centred local Taylor expansion at particle positions.
It is the final far-field operator in a conventional FMM. We first build a
valid local expansion with P2M and M2L, then compare its spatial error across
three planes through the local centre: XY, XZ, and an oblique diagonal plane
that varies all three Cartesian coordinates.

## Mathematical definition

For $dx=x-c_t$,

$$\phi_p(x)=\sum_{|\beta|\le p}L_\beta\frac{dx^\beta}{\beta!},
\qquad H_p(x)=-\nabla\phi_p(x).$$

The local expansion is most accurate near $c_t$. Its valid region depends on
the separation from the source cluster and on truncation order.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import cdfmm

try:
    from example_utils import (
        direct_fields,
        draw_box_3d,
        error_metrics,
        finish_3d_axes,
        local_fields,
        multipole_fields,
        new_3d_figure,
        nodes_at_level,
        plot_coefficients_by_degree,
        random_unit_vectors,
        relative_error,
        set_axes_equal,
        vec3_to_array,
        zoom_3d_axes,
    )
except ModuleNotFoundError:
    # This path is used when the kernel starts in the repository root.
    from examples.notebooks.example_utils import (
        direct_fields,
        draw_box_3d,
        error_metrics,
        finish_3d_axes,
        local_fields,
        multipole_fields,
        new_3d_figure,
        nodes_at_level,
        plot_coefficients_by_degree,
        random_unit_vectors,
        relative_error,
        set_axes_equal,
        vec3_to_array,
        zoom_3d_axes,
    )

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True})

## User parameters

In [ ]:
expansion_order = 5
n_sources = 80
random_seed = 42
source_centre = np.zeros(3)
local_centre = np.array([4.0, 0.5, 0.0])
plane_half_width = 0.45
grid_points_per_axis = 23

## Build the local expansion

In [ ]:
rng = np.random.default_rng(random_seed)
source_positions = rng.uniform(-0.4, 0.4, size=(n_sources, 3))
dipole_moments = rng.normal(size=(n_sources, 3))

multipole_coefficients = cdfmm.p2m_dipole(
    source_centre,
    source_positions,
    dipole_moments,
    order=expansion_order,
)
local_coefficients = cdfmm.m2l(
    multipole_coefficients,
    source_centre,
    local_centre,
    order=expansion_order,
)

print(f"Expansion order: {expansion_order}")
print(f"Coefficient count: {len(local_coefficients)}")
print(f"Source/local separation: {np.linalg.norm(local_centre - source_centre):.3f}")

## Evaluate three target planes

All planes use the same square coordinates and resolution, so their error
statistics are directly comparable. The diagonal plane is perpendicular to
$(1,1,1)$ and uses two orthonormal in-plane directions; moving across it
therefore changes x, y, and z rather than holding one Cartesian coordinate
fixed.

In [ ]:
offset_coordinates = np.linspace(
    -plane_half_width,
    plane_half_width,
    grid_points_per_axis,
)
u_offsets, v_offsets = np.meshgrid(offset_coordinates, offset_coordinates)

diagonal_u = np.array([1.0, -1.0, 0.0]) / np.sqrt(2.0)
diagonal_v = np.array([1.0, 1.0, -2.0]) / np.sqrt(6.0)

plane_definitions = {
    "xy": {
        "title": "XY plane (z centred)",
        "u": np.array([1.0, 0.0, 0.0]),
        "v": np.array([0.0, 1.0, 0.0]),
        "u_label": "x offset",
        "v_label": "y offset",
        "colour": "tab:red",
    },
    "xz": {
        "title": "XZ plane (y centred)",
        "u": np.array([1.0, 0.0, 0.0]),
        "v": np.array([0.0, 0.0, 1.0]),
        "u_label": "x offset",
        "v_label": "z offset",
        "colour": "tab:purple",
    },
    "diagonal": {
        "title": "Diagonal plane (normal [1,1,1])",
        "u": diagonal_u,
        "v": diagonal_v,
        "u_label": "diagonal u offset",
        "v_label": "diagonal v offset",
        "colour": "tab:cyan",
    },
}

plane_results = {}
for plane_name, definition in plane_definitions.items():
    offsets = (
        u_offsets[..., np.newaxis] * definition["u"]
        + v_offsets[..., np.newaxis] * definition["v"]
    )
    target_positions = local_centre + offsets.reshape(-1, 3)

    reference_fields = direct_fields(
        target_positions,
        source_positions,
        dipole_moments,
    )
    l2p_plane_fields = local_fields(
        target_positions,
        local_coefficients,
        local_centre,
        expansion_order,
    )
    m2p_plane_fields = multipole_fields(
        target_positions,
        multipole_coefficients,
        source_centre,
        expansion_order,
    )

    plane_results[plane_name] = {
        "targets": target_positions,
        "reference_fields": reference_fields,
        "l2p_fields": l2p_plane_fields,
        "m2p_fields": m2p_plane_fields,
        "l2p_errors": relative_error(
            l2p_plane_fields,
            reference_fields,
        ).reshape(u_offsets.shape),
        "m2p_errors": relative_error(
            m2p_plane_fields,
            reference_fields,
        ).reshape(u_offsets.shape),
        "l2p_metrics": error_metrics(l2p_plane_fields, reference_fields),
        "m2p_metrics": error_metrics(m2p_plane_fields, reference_fields),
    }

print("Plane       method                    mean          RMS       maximum")
print("-----------------------------------------------------------------------")
for plane_name in plane_definitions:
    for method_name in ["l2p", "m2p"]:
        metrics = plane_results[plane_name][f"{method_name}_metrics"]
        route = "P2M + M2L + L2P" if method_name == "l2p" else "P2M + M2P"
        print(
            f"{plane_name:10s}  {route:20s}  "
            f"{metrics['mean']:10.3e}  {metrics['rms']:10.3e}  "
            f"{metrics['maximum']:10.3e}"
        )


## Geometry of the three L2P target planes

The orange source centre and blue particles are distant from the green local
centre. The three coloured target grids pass through the same local centre but
have different orientations. The overview shows their separation from the
source; the next figure zooms into both regions independently.

In [ ]:
def draw_plane_boundary(axes, centre, basis_u, basis_v, half_width, colour):
    """Draw the four edges of one square target plane."""
    corners = np.array(
        [
            centre - half_width * basis_u - half_width * basis_v,
            centre + half_width * basis_u - half_width * basis_v,
            centre + half_width * basis_u + half_width * basis_v,
            centre - half_width * basis_u + half_width * basis_v,
            centre - half_width * basis_u - half_width * basis_v,
        ]
    )
    axes.plot(*corners.T, color=colour, linewidth=1.5)


figure, axes = new_3d_figure(figsize=(10, 7.5))
axes.scatter(
    *source_positions.T,
    s=10,
    color="tab:blue",
    alpha=0.55,
    label="sources",
)
axes.scatter(
    *source_centre,
    marker="D",
    s=100,
    color="tab:orange",
    label="source multipole centre",
)
axes.scatter(
    *local_centre,
    marker="D",
    s=110,
    color="tab:green",
    label="local expansion centre",
)

for plane_name, definition in plane_definitions.items():
    target_positions = plane_results[plane_name]["targets"]
    axes.scatter(
        *target_positions[::16].T,
        marker=".",
        s=14,
        color=definition["colour"],
        alpha=0.65,
        label=definition["title"],
    )
    draw_plane_boundary(
        axes,
        local_centre,
        definition["u"],
        definition["v"],
        plane_half_width,
        definition["colour"],
    )

finish_3d_axes(axes, "Source cluster and three L2P target planes")
axes.legend(fontsize=8)
figure.tight_layout()


### Close-up views of the source and L2P target regions

The source close-up resolves the dipoles compressed by P2M. The target
close-up removes the large source-to-local separation and shows how the XY,
XZ, and diagonal grids intersect at the local centre.

In [ ]:
figure = plt.figure(figsize=(13, 5.5))
source_axes = figure.add_subplot(121, projection="3d")
target_axes = figure.add_subplot(122, projection="3d")

source_axes.scatter(
    *source_positions.T,
    s=22,
    color="tab:blue",
    alpha=0.7,
    label="dipole sources",
)
source_axes.scatter(
    *source_centre,
    marker="D",
    s=105,
    color="tab:orange",
    label="multipole centre",
)
zoom_3d_axes(source_axes, source_centre, 0.55, "Source-region close-up")
source_axes.legend(fontsize=8)

for plane_name, definition in plane_definitions.items():
    target_positions = plane_results[plane_name]["targets"]
    target_axes.scatter(
        *target_positions[::12].T,
        marker=".",
        s=18,
        color=definition["colour"],
        alpha=0.7,
        label=definition["title"],
    )
    draw_plane_boundary(
        target_axes,
        local_centre,
        definition["u"],
        definition["v"],
        plane_half_width,
        definition["colour"],
    )

target_axes.scatter(
    *local_centre,
    marker="D",
    s=120,
    color="tab:green",
    label="local expansion centre",
)
zoom_3d_axes(
    target_axes,
    local_centre,
    1.2 * plane_half_width,
    "Three target planes at the local centre",
)
target_axes.legend(fontsize=7)
figure.tight_layout()


## Spatial error heatmaps

Columns correspond to the three planes. The top row evaluates the local route
and the bottom row evaluates M2P directly from the same source multipole.
Every panel uses the same colour limits, and each colour bar is attached only
to its own axes so it cannot overlap a neighbouring plot.

In [ ]:
all_log_errors = []
for plane_name in plane_definitions:
    for method_name in ["l2p", "m2p"]:
        errors = plane_results[plane_name][f"{method_name}_errors"]
        all_log_errors.append(np.log10(np.maximum(errors, 1.0e-18)))

colour_minimum = min(values.min() for values in all_log_errors)
colour_maximum = max(values.max() for values in all_log_errors)

figure, axes = plt.subplots(
    2,
    3,
    figsize=(17, 9.5),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

for column, (plane_name, definition) in enumerate(plane_definitions.items()):
    for row, (method_name, route_title) in enumerate(
        [
            ("l2p", "P2M + M2L + L2P"),
            ("m2p", "P2M + M2P"),
        ]
    ):
        current_axes = axes[row, column]
        log_errors = np.log10(
            np.maximum(
                plane_results[plane_name][f"{method_name}_errors"],
                1.0e-18,
            )
        )
        image = current_axes.imshow(
            log_errors,
            origin="lower",
            extent=[
                -plane_half_width,
                plane_half_width,
                -plane_half_width,
                plane_half_width,
            ],
            vmin=colour_minimum,
            vmax=colour_maximum,
            cmap="magma",
            aspect="equal",
        )
        current_axes.scatter(
            0.0,
            0.0,
            marker="x",
            color="cyan",
            label="local centre",
        )
        current_axes.set_xlabel(definition["u_label"])
        current_axes.set_ylabel(definition["v_label"])
        current_axes.set_title(f"{definition['title']}\n{route_title}")
        current_axes.legend(loc="upper right", fontsize=7)

        # Give every panel its own colour bar with reserved layout space.
        colour_bar = figure.colorbar(
            image,
            ax=current_axes,
            fraction=0.046,
            pad=0.04,
        )
        colour_bar.set_label(r"$\log_{10}$ relative field error")

figure.suptitle("Spatial accuracy across three target-plane orientations")


## Plane-to-plane comparison

The grouped RMS plot condenses each heatmap into one value. This makes
orientation-dependent differences visible while retaining the heatmaps for
their spatial structure.

In [ ]:
plane_names = list(plane_definitions)
plane_labels = [plane_definitions[name]["title"] for name in plane_names]
l2p_rms_errors = [
    plane_results[name]["l2p_metrics"]["rms"]
    for name in plane_names
]
m2p_rms_errors = [
    plane_results[name]["m2p_metrics"]["rms"]
    for name in plane_names
]

bar_positions = np.arange(len(plane_names))
bar_width = 0.36
figure, axes = plt.subplots(figsize=(9, 5))
axes.bar(
    bar_positions - 0.5 * bar_width,
    l2p_rms_errors,
    width=bar_width,
    label="P2M + M2L + L2P",
    color="tab:green",
)
axes.bar(
    bar_positions + 0.5 * bar_width,
    m2p_rms_errors,
    width=bar_width,
    label="P2M + M2P",
    color="tab:orange",
)
axes.set_yscale("log")
axes.set_xticks(bar_positions, plane_labels, rotation=12, ha="right")
axes.set_ylabel("RMS relative field error")
axes.set_title("Error comparison across target-plane orientations")
axes.legend()
figure.tight_layout()


## What to observe

The local Taylor representation is generally most accurate near its expansion
centre, with error increasing toward each plane boundary. Comparing XY, XZ,
and diagonal planes reveals directional variation that one Cartesian slice
cannot show. M2P has a different spatial pattern because it expands about the
source centre. All three experiments use identical physical half-widths,
sampling density, source data, coefficients, and direct P2P references.